In [1]:
import os
import geopandas as gpd
import folium
from IPython.display import HTML, display
import ipywidgets as widgets

SHAPE_DIR = "CrimeToolHotspots"

In [2]:
# Collect all .shp files
shp_files = [f for f in os.listdir(SHAPE_DIR) if f.endswith(".shp")]

crime_types = set()
quarters = set()

for f in shp_files:
    # Example: AlcoholRelatedAssault_2025_Q3.shp
    name = f.replace(".shp", "")
    parts = name.split("_")
    # Last two parts: year + quarter (e.g. 2025, Q3)
    crime = "_".join(parts[:-2])
    year = parts[-2]
    quarter = parts[-1]  # e.g. Q3

    crime_types.add(crime)
    quarters.add(f"{year}_{quarter}")

crime_types = sorted(list(crime_types))
quarters = sorted(list(quarters))

In [3]:
crime_dropdown = widgets.Dropdown(
    options=crime_types,
    description="Crime:",
    layout=widgets.Layout(width="300px")
)

quarter_dropdown = widgets.Dropdown(
    options=quarters,
    description="Period:",
    layout=widgets.Layout(width="200px")
)

In [9]:
def load_shapefile(crime, period):
    year, q = period.split("_") 
    filename = f"{crime}_{year}_{q}.shp"
    path = os.path.join(SHAPE_DIR, filename)
    gdf = gpd.read_file(path)
    return gdf

def make_map(gdf):
    # Center map
    center = gdf.geometry.union_all().centroid

    m = folium.Map(location=[center.y, center.x], zoom_start=7)

    # Add polygons (no choropleth)
    folium.GeoJson(
        gdf,
        name="Hotspots",
        style_function=lambda x: {
            "fillColor": "#ff0000",
            "color": "black",
            "weight": 1,
            "fillOpacity": 0.4
        },
        tooltip=folium.GeoJsonTooltip(
            fields=[c for c in ["LGA_NAME", "LGA", "NAME"] if c in gdf.columns]
        )
    ).add_to(m)

    return m

In [10]:
import webbrowser

def update_map(crime, period):
    gdf = load_shapefile(crime, period)
    m = make_map(gdf)

    # Save to HTML
    filepath = os.path.abspath("CrimeMap.html")
    m.save(filepath)

    # Show clickable link
    display(HTML(f"<h3><a href='file://{filepath}' target='_blank'>👉 Open Crime Map</a></h3>"))

    # Force Chrome specifically
    chrome_path = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"
    if os.path.exists(chrome_path):
        webbrowser.get(chrome_path).open("file://" + filepath)
    else:
        webbrowser.open("file://" + filepath)

In [11]:
os.path.abspath("CrimeMap.html")

'/Users/Surface/Documents/GitHub/CrimeStopper/CrimeMap.html'

In [8]:
gdf = load_shapefile("Robbery", "2025_Q3")
gdf.columns

Index(['Density', 'geometry'], dtype='object')

In [13]:
m